In [4]:
import os
import sys
from pathlib import Path

ROOT = Path(os.path.abspath('')).resolve().parents[2]
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import nvitk.core as core
core.setup(globals())

import nvitk as nv
from nvitk.core import as_backend_array
from nvitk.pipes.qvtpy.util.vessel_cd_segmentation import VesselSegStats, LocalSegResult
from nvitk.pipes.qvtpy.util.vessel_cd_segmentation import _bbox_with_padding, _threshold_crop, _paste_crop_mask, _region_grow_vessel


In [14]:
def build_seg_4dflow_local(
    cd: np.ndarray,
    centerlines_mask: np.ndarray,
    *,
    crop_padding_bbox: int = 0,
    thr_algorithm = "lsthr",
    region_growing: bool = True,
    rg_intensity_frac: float = 0.5,
):
    """Build multilabel ``seg_4dflow`` from CD and per-label centerline backbone."""
    cd = as_backend_array(cd).astype(np.float64)
    clm = as_backend_array(centerlines_mask).astype(np.int32, copy=False)
    shape = tuple(int(s) for s in clm.shape[:3])
    seg = np.zeros(shape, dtype=np.int32)

    label_ids = sorted(int(v) for v in np.unique(clm) if int(v) > 0)
    stats = []
    opt_thresh_by_label: dict[int, float | None] = {}

    for lid in label_ids:
        roi = clm == lid
        bbox = _bbox_with_padding(roi, shape, padding=crop_padding_bbox)
        print(bbox)
        if bbox is None:
            stats.append(
                VesselSegStats(
                    label_id=lid,
                    bbox=(0, 0, 0, 0, 0, 0),
                    thr_algorithm=thr_algorithm,
                    opt_thresh=None,
                    n_voxels_after_threshold=0,
                    n_voxels_after_region_growing=0,
                    warning="empty centerline mask for label",
                )
            )
            opt_thresh_by_label[lid] = None
            continue

        i0, i1, j0, j1, k0, k1 = bbox
        cd_crop = cd[i0 : i1 + 1, j0 : j1 + 1, k0 : k1 + 1]
        nv.imsave(f'cd_crop_{lid}.nii.gz', cd_crop, metadata=centerlines_mask.metadata)
        crop_mask, opt_t, warn = _threshold_crop(cd_crop, thr_algorithm)
        nv.imsave(f'crop_mask_{lid}.nii.gz', crop_mask.astype(np.uint8), metadata=centerlines_mask.metadata)
        opt_thresh_by_label[lid] = opt_t
        n_thr = _paste_crop_mask(seg, crop_mask, lid, bbox)
        nv.imsave(f'seg_{lid}.nii.gz', seg.astype(np.uint8), metadata=centerlines_mask.metadata)

        stats.append(
            VesselSegStats(
                label_id=lid,
                bbox=bbox,
                thr_algorithm=thr_algorithm,
                opt_thresh=opt_t,
                n_voxels_after_threshold=n_thr,
                n_voxels_after_region_growing=n_thr,
                warning=warn,
            )
        )

    if region_growing:
        for st in stats:
            lid = st.label_id
            floor = opt_thresh_by_label.get(lid)
            _region_grow_vessel(
                seg,
                cd,
                lid,
                rg_intensity_frac=rg_intensity_frac,
                rg_abs_floor=floor,
            )
            st.n_voxels_after_region_growing = int(np.count_nonzero(seg == lid))

    return LocalSegResult(
        segmentation=as_backend_array(seg.astype(np.int32, copy=False)),
        vessel_stats=stats,
    )

In [15]:
cd = nv.imread('/home/imarcoss/NetVolumes/LAB_MCC/LabVF/PESA-Brain/DATA/NIFTI/PESA5745609/4DFlow/ComplexDifference_3D.nii.gz')
clm = nv.imread('/home/imarcoss/NetVolumes/LAB_MCC/LabVF/PESA-Brain/RESULTS/res_QVTPy/PESA5745609/qvtpy/stage3_centerline/centerlines_mask.nii.gz')
seg = build_seg_4dflow_local(cd, clm, region_growing=False)

(108, 118, 147, 167, 20, 41)
(138, 150, 147, 167, 19, 39)
(125, 135, 134, 138, 12, 40)
(110, 124, 146, 154, 46, 50)
(132, 143, 149, 151, 44, 46)
(94, 102, 147, 149, 43, 45)
(153, 159, 151, 153, 41, 42)
(132, 141, 138, 146, 35, 41)
(105, 114, 118, 140, 37, 43)
(135, 154, 115, 136, 36, 44)
(118, 122, 139, 139, 31, 35)
(128, 136, 133, 133, 39, 41)
(131, 139, 18, 38, 10, 97)
(128, 130, 49, 84, 15, 67)
(53, 100, 43, 74, 4, 21)
(152, 205, 43, 83, 6, 20)
